In [ ]:
# Synthetic Direct Postgres → Bronze Handoff

This notebook skips the Kafka staging and long-format melting path.

It takes the wide synthetic stream table already stored in Postgres and turns it into a single bronze-ready table that looks like the original pump dataset shape.

## What this notebook does
1. Reads one or more synthetic batches from Postgres.
2. Combines them into one stable ordered table.
3. Creates a unified row number and unified episode number.
4. Adds a fresh time index and timestamp series.
5. Maps synthetic state values into the original-style `machine_status` label.
6. Cuts the table down to the final bronze handoff structure.
7. Optionally writes the final result back to Postgres and/or parquet.

## Notes before you run this

- This notebook assumes your source table is already **wide** and contains the synthetic sensor columns plus metadata such as `batch_id`, `stream_state`, `phase`, and `meta__episode_id`.
- It is designed to handle **multiple appended batches** where episode numbering may restart inside each batch.
- The final dataframe can be kept in an **original-dataset style** (`timestamp`, `sensor_*`, `machine_status`) or keep extra lineage columns at the end for Bronze auditing.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

# Try a few likely roots so the notebook can still import the helper module
# if you place it in your project notebook folder structure.
candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path.cwd().parent.parent.parent,
]

for root in candidate_roots:
    if (root / "utils").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from utils.synthetic_postgres_to_bronze import (
    build_postgres_engine_from_env,
    table_exists,
    get_table_columns,
    get_sensor_columns,
    read_synthetic_stream_table,
    build_bronze_ready_from_postgres,
    summarize_bronze_ready_dataframe,
    write_dataframe_to_postgres,
)

## Configuration

Update these values to match your current synthetic source table and your desired bronze output table.

In [ ]:
# ---------------------------
# Source table settings
# ---------------------------
DATASET_NAME = "pump"

SOURCE_SCHEMA = "capstone"
SOURCE_TABLE = f"synthetic_{DATASET_NAME}_stream"

# Optional: set to a list such as [1, 2, 3] to only use selected batches.
BATCH_IDS = None

# ---------------------------
# New unified time settings
# ---------------------------
START_TIMESTAMP = "2018-04-01 00:00:00"
SAMPLING_FREQUENCY = "1min"

# ---------------------------
# Final output settings
# ---------------------------
KEEP_LINEAGE_COLUMNS = True

# Optional row cutdown after the table is unified.
# Set to None to keep all rows.
TARGET_TOTAL_ROWS = None
TRIM_MODE = "head"   # head | tail | random

# ---------------------------
# Write-back settings
# ---------------------------
WRITE_TO_POSTGRES = True
TARGET_SCHEMA = "capstone"
TARGET_TABLE = f"bronze_{DATASET_NAME}_direct_handoff"
TARGET_IF_EXISTS = "replace"   # replace | append

# Optional exports
EXPORT_PARQUET = False
PARQUET_OUTPUT_PATH = Path("artifacts") / "synthetic" / DATASET_NAME / f"{TARGET_TABLE}.parquet"

EXPORT_CSV = False
CSV_OUTPUT_PATH = Path("artifacts") / "synthetic" / DATASET_NAME / f"{TARGET_TABLE}.csv"

## Connect to Postgres and inspect the source table

In [ ]:
engine = build_postgres_engine_from_env()

print("Source table exists:", table_exists(engine, schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE))

source_columns = get_table_columns(engine, schema=SOURCE_SCHEMA, table_name=SOURCE_TABLE)
print("Source column count:", len(source_columns))
print("Sensor column count:", len(get_sensor_columns(source_columns)))
print("First 20 columns:", source_columns[:20])

## Preview the raw source rows

This cell lets you confirm that the table already looks like the wide synthetic stream you expect before the bronze handoff logic runs.

In [ ]:
raw_preview_df = read_synthetic_stream_table(
    engine,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE,
    batch_ids=BATCH_IDS,
)

print("Raw selected row count:", len(raw_preview_df))
print("Raw selected column count:", len(raw_preview_df.columns))

display(raw_preview_df.head())

## Build the bronze-ready dataframe

This is the main conversion step.

In [ ]:
bronze_ready_df = build_bronze_ready_from_postgres(
    engine,
    source_schema=SOURCE_SCHEMA,
    source_table=SOURCE_TABLE,
    batch_ids=BATCH_IDS,
    start_timestamp=START_TIMESTAMP,
    frequency=SAMPLING_FREQUENCY,
    keep_lineage_columns=KEEP_LINEAGE_COLUMNS,
    target_total_rows=TARGET_TOTAL_ROWS,
    trim_mode=TRIM_MODE,
)

bronze_summary = summarize_bronze_ready_dataframe(bronze_ready_df)

print(json.dumps(bronze_summary, indent=2))

## Review the final result

In [ ]:
print("Final row count:", len(bronze_ready_df))
print("Final column count:", len(bronze_ready_df.columns))
print("Final columns:")
print(bronze_ready_df.columns.tolist())

display(bronze_ready_df.head())

## Status distribution and batch coverage checks

In [ ]:
if "machine_status" in bronze_ready_df.columns:
    print("machine_status counts:")
    print(bronze_ready_df["machine_status"].value_counts(dropna=False))

if "batch_id" in bronze_ready_df.columns:
    print("\nrows by batch_id:")
    print(bronze_ready_df["batch_id"].value_counts().sort_index())

if "meta__episode_id_unified" in bronze_ready_df.columns:
    print("\nunified episode count:", bronze_ready_df["meta__episode_id_unified"].nunique())

## Optional: export parquet / csv

In [ ]:
if EXPORT_PARQUET:
    PARQUET_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    bronze_ready_df.to_parquet(PARQUET_OUTPUT_PATH, index=False)
    print("Parquet written to:", PARQUET_OUTPUT_PATH)

if EXPORT_CSV:
    CSV_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    bronze_ready_df.to_csv(CSV_OUTPUT_PATH, index=False)
    print("CSV written to:", CSV_OUTPUT_PATH)

if not EXPORT_PARQUET and not EXPORT_CSV:
    print("No file export selected.")

## What to hand into Bronze next

At this point, `bronze_ready_df` should already be in the shape you want for a Bronze-style ingest handoff.

Typical next use:
- keep only original dataset columns if you want a very faithful raw handoff
- or keep the lineage columns for auditability while you test the new synthetic path